In [92]:
import pandas as pd
import sklearn
import torch
from datetime import datetime

# Feature Data

In [93]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../data/premier_league/'
all_data_df={}
all_target_df={}
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_data_df[season]=pd.read_csv(f'{season_path}/all_data_df.csv')
    all_data_df[season]['date']=all_data_df[season]['date'].apply(lambda s: datetime.strptime(s, '%B-%d-%Y'))
    all_data_df[season]=all_data_df[season].sort_values('date').reset_index(drop=True)

    all_target_df[season]=pd.read_csv(f'{season_path}/all_target_df.csv')
    all_target_df[season]['date']=all_target_df[season]['date'].apply(lambda s: datetime.strptime(s, '%B-%d-%Y'))
    all_target_df[season]=all_target_df[season].sort_values('date').reset_index(drop=True)

In [94]:
for season in seasons:
    all_teams=all_data_df[season]['home'].unique().tolist()
    all_team_date_dff=[]
    team_temp_counter={}
    for team in all_teams:
        team_temp_counter[team]=0
        team_data_df=all_data_df[season][(all_data_df[season]['home']==team)|(all_data_df[season]['away']==team)]
        date_diff=team_data_df['date'].iloc[1:].reset_index(drop=True)-team_data_df['date'].iloc[:-1].reset_index(drop=True)
        date_diff=date_diff.apply(lambda x: x.days)
        date_diff.index=date_diff.index+1
        date_diff[0]=-1
        date_diff=date_diff.sort_index()
        all_team_date_dff.append(date_diff)
    all_team_date_dff=pd.concat(all_team_date_dff, axis=1)
    all_team_date_dff.columns=all_teams

    home_day_diff, away_day_diff=[], []
    for _, row in all_data_df[season].iterrows():
        home, away=row['home'], row['away']
        home_day_diff.append(all_team_date_dff.loc[team_temp_counter[home], home])
        team_temp_counter[home]+=1
        away_day_diff.append(all_team_date_dff.loc[team_temp_counter[away], away])
        team_temp_counter[away]+=1
    all_data_df[season]['home_day_diff']=home_day_diff
    all_data_df[season]['away_day_diff']=away_day_diff
    #all_data_df[season]['date_since_last_match']=date_diff
    #all_data_df[season]=all_data_df[season].drop(columns=['date'])
    

In [100]:
all_data_df['2023-24'].head(10)

,home_performance_pk,home_performance_pkatt,home_performance_crdr,home_performance_touches,home_performance_tkl,home_performance_int,home_performance_blocks,home_expected_xg,home_expected_npxg,home_expected_xag,...,away_performance_og,away_performance_recov,away_aerial_duels_won,away_aerial_duels_lost,away_aerial_duels_won%,home,away,date,home_day_diff,away_day_diff
0,0,0,1,496,12,7,12,0.3,0.3,0.3,...,0,54,13,13,50.0,Burnley,Manchester City,2023-08-11,-1,-1
1,0,0,0,693,17,7,17,1.4,1.4,0.3,...,0,52,29,16,64.4,Bournemouth,West Ham,2023-08-12,-1,-1
2,0,0,0,646,16,7,12,3.4,3.4,3.3,...,0,44,6,5,54.5,Newcastle Utd,Aston Villa,2023-08-12,-1,-1
3,0,0,0,902,19,7,6,0.8,0.8,0.6,...,0,34,20,12,62.5,Arsenal,Nott'ham Forest,2023-08-12,-1,-1
4,0,0,0,441,22,11,23,0.5,0.5,0.4,...,0,55,31,13,70.5,Sheffield Utd,Crystal Palace,2023-08-12,-1,-1
5,0,0,0,531,13,13,13,2.7,2.7,2.4,...,0,43,14,9,60.9,Everton,Fulham,2023-08-12,-1,-1
6,1,1,0,769,13,4,8,4.0,3.2,2.9,...,0,41,21,12,63.6,Brighton,Luton Town,2023-08-12,-1,-1
7,1,1,0,464,17,8,15,2.2,1.4,1.3,...,0,50,11,15,42.3,Brentford,Tottenham,2023-08-13,-1,-1
8,0,0,0,862,13,7,14,1.4,1.4,1.3,...,0,62,6,11,35.3,Chelsea,Liverpool,2023-08-13,-1,-1
9,0,0,0,643,18,8,21,2.2,2.2,2.2,...,0,69,17,7,70.8,Manchester Utd,Wolves,2023-08-14,-1,-1


In [110]:
all_data_df['2023-24']['home']+'-'+all_data_df['2023-24']['away']+'-'+all_data_df['2023-24']['date'].apply(lambda x: datetime.strftime(x, "%m/%d/%Y"))

0           Burnley-Manchester City-08/11/2023
1              Bournemouth-West Ham-08/12/2023
2         Newcastle Utd-Aston Villa-08/12/2023
3           Arsenal-Nott'ham Forest-08/12/2023
4      Sheffield Utd-Crystal Palace-08/12/2023
                        ...                   
375                Liverpool-Wolves-05/19/2024
376         Brighton-Manchester Utd-05/19/2024
377             Chelsea-Bournemouth-05/19/2024
378               Luton Town-Fulham-05/19/2024
379         Burnley-Nott'ham Forest-05/19/2024
Length: 380, dtype: object

In [121]:
feature_df=pd.concat((all_data_df.values()), axis=0)
feature_df=feature_df.sort_values('date')
feature_df.index=feature_df['home']+'-'+feature_df['away']+'-'+feature_df['date'].apply(lambda x: datetime.strftime(x, "%m/%d/%Y"))

In [122]:
feature_df

,home_performance_pk,home_performance_pkatt,home_performance_crdr,home_performance_touches,home_performance_tkl,home_performance_int,home_performance_blocks,home_expected_xg,home_expected_npxg,home_expected_xag,...,away_performance_og,away_performance_recov,away_aerial_duels_won,away_aerial_duels_lost,away_aerial_duels_won%,home,away,date,home_day_diff,away_day_diff
Liverpool-Norwich City-08/09/2019,0,0,0,627,21,14,11,1.8,1.8,1.6,...,1,37,7,15,31.8,Liverpool,Norwich City,2019-08-09,-1,-1
Watford-Brighton-08/10/2019,0,0,0,562,12,12,11,0.7,0.7,0.4,...,0,56,18,14,56.3,Watford,Brighton,2019-08-10,-1,-1
Tottenham-Aston Villa-08/10/2019,0,0,0,749,17,5,9,2.5,2.5,1.5,...,0,39,12,10,54.5,Tottenham,Aston Villa,2019-08-10,-1,-1
Crystal Palace-Everton-08/10/2019,0,0,0,427,21,10,16,0.9,0.9,0.9,...,0,58,18,30,37.5,Crystal Palace,Everton,2019-08-10,-1,-1
Burnley-Southampton-08/10/2019,0,0,0,507,21,14,14,0.9,0.9,0.6,...,0,53,22,22,50.0,Burnley,Southampton,2019-08-10,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Everton-Crystal Palace-09/28/2024,0,0,0,509,14,16,11,0.9,0.9,0.8,...,0,51,20,28,41.7,Everton,Crystal Palace,2024-09-28,7,7
Nott'ham Forest-Fulham-09/28/2024,0,0,0,494,8,9,13,0.8,0.8,0.6,...,0,29,7,18,28.0,Nott'ham Forest,Fulham,2024-09-28,6,7
Manchester Utd-Tottenham-09/29/2024,0,0,1,554,27,17,14,1.0,1.0,0.8,...,0,46,8,6,57.1,Manchester Utd,Tottenham,2024-09-29,8,8
Ipswich Town-Aston Villa-09/29/2024,0,0,0,527,18,7,5,1.2,1.2,1.1,...,0,33,7,8,46.7,Ipswich Town,Aston Villa,2024-09-29,8,8


In [123]:
feature_df=pd.concat([feature_df, pd.get_dummies(feature_df['home']).rename(columns=lambda x: f'home_{x}')], axis=1)
feature_df=pd.concat([feature_df, pd.get_dummies(feature_df['away']).rename(columns=lambda x: f'away_{x}')], axis=1)

In [124]:
feature_df=feature_df.drop(columns=['home', 'away'])
feature_df=feature_df.drop(columns='date')

# Target Processing

In [118]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../data/premier_league/'
all_targer_df={}
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_targer_df[season]=pd.read_csv(f'{season_path}/all_target_df.csv')

In [125]:
all_targer_df=pd.concat(all_targer_df.values(), axis=0)

In [126]:
all_targer_df.index=all_targer_df['home']+'-'+all_targer_df['away']+'-'+all_targer_df['date'].apply(lambda x: datetime.datetime.strftime(x, "%m/%d/%Y"))
all_targer_df=all_targer_df.drop(columns=['home', 'away'])
all_targer_df=all_targer_df.drop(columns='date')

TypeError: descriptor 'strftime' for 'datetime.date' objects doesn't apply to a 'str' object

In [ ]:
target_config={
    'home_goals': (0,4,1),
    'away_goals': (0,4,1),
    'home_corners': (5,10,1),
    'away_corners': (5,10,1),
    'home_cards': (0,5,1),
    'away_cards': (0,5,1),
    'home_shots': (5,20,2),
    'away_shots': (5,20,2),
    'home_sots': (0,10,1),
    'away_sots':(0,10,1),
}